In [ ]:
from typing import Any, List, Optional, Tuple
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

def stack_feature_columns_matrix(
    df: DataFrame,
    feature_columns_1: List[str],
    feature_columns_2: List[str],
    baseline_list: List[str],
    perf_list: List[str],
    seg_list: List[str],
) -> DataFrame:
    """
    Stack two score lists and create all score_1 x score_2 combinations.
    """
    baseline_number = len(baseline_list)
    perf_number = len(perf_list)
    seg_number = len(seg_list)
    feature_number_1 = len(feature_columns_1)
    feature_number_2 = len(feature_columns_2)

    stack_expression_1 = ", ".join([f"'{c}', cast(`{c}` as double)" for c in feature_columns_1])
    stack_expression_2 = ", ".join([f"'{c}', cast(`{c}` as double)" for c in feature_columns_2])
    stack_perf = ", ".join([f"'{c}', cast(`{c}` as double)" for c in perf_list])
    stack_seg = ", ".join([f"'{c}', cast(`{c}` as string)" for c in seg_list])
    baseline_seg = ", ".join([f"'{c}', cast(`{c}` as string)" for c in baseline_list])

    stacked = df.select(
        "*",
        F.expr(f"stack({baseline_number}, {baseline_seg}) as (baseline_name, baseline_value)"),
        F.expr(f"stack({perf_number}, {stack_perf}) as (perf_name, perf_value)"),
        F.expr(f"stack({seg_number}, {stack_seg}) as (seg_name, seg_value)"),
        F.expr(f"stack({feature_number_1}, {stack_expression_1}) as (score_name_1, score_value_1)"),
        F.expr(f"stack({feature_number_2}, {stack_expression_2}) as (score_name_2, score_value_2)"),
    )

    return (
        stacked
        .withColumn("score_value_1", F.round(F.col("score_value_1"), 4))
        .withColumn("score_value_2", F.round(F.col("score_value_2"), 4))
    )


def _calculate_numerical_threshold_bounds_matrix(
    dev_df: DataFrame,
    bins: int,
) -> Tuple[DataFrame, DataFrame]:
    """
    Calculate lower and upper bounds for score_1 and score_2 separately.
    """
    partition_by = [
        "score_name_1",
        "score_name_2",
        "perf_name",
        "seg_name",
        "seg_value",
        "baseline_name",
        "baseline_value",
    ]

    prct_rank_1 = dev_df.select(
        *partition_by,
        "score_value_1",
        F.percent_rank().over(Window.partitionBy(*partition_by).orderBy("score_value_1")).alias(
            "prct_rank"
        ),
    )

    bounds_1 = (
        prct_rank_1
        .withColumn("bins_", F.ceil(F.col("prct_rank") * bins))
        .withColumn("band_1", F.when(F.col("bins_") == 0, 1).otherwise(F.col("bins_")))
        .groupBy(*partition_by, "band_1")
        .agg(
            F.min("score_value_1").alias("lower_bound_1"),
            F.max("score_value_1").alias("upper_bound_1"),
        )
    )

    partition_by_1 = Window.partitionBy(*partition_by).orderBy("band_1")
    bounds_1 = (
        bounds_1
        .withColumn(
            "new_lower_bound_incomplete_1",
            F.lag("upper_bound_1", 1, -float('inf')).over(partition_by_1),
        )
        .withColumn(
            "upper_bound_1",
            F.lag("new_lower_bound_incomplete_1", -1, float('inf')).over(partition_by_1),
        )
        .withColumn(
            "lower_bound_1",
            F.coalesce(F.col("new_lower_bound_incomplete_1"), F.col("lower_bound_1")),
        )
        .drop("new_lower_bound_incomplete_1")
    )

    prct_rank_2 = dev_df.select(
        *partition_by,
        "score_value_2",
        F.percent_rank().over(Window.partitionBy(*partition_by).orderBy("score_value_2")).alias(
            "prct_rank"
        ),
    )

    bounds_2 = (
        prct_rank_2.withColumn("bins_", F.ceil(F.col("prct_rank") * bins))
        .withColumn("band_2", F.when(F.col("bins_") == 0, 1).otherwise(F.col("bins_")))
        .groupBy(*partition_by, "band_2")
        .agg(
            F.min("score_value_2").alias("lower_bound_2"),
            F.max("score_value_2").alias("upper_bound_2"),
        )
    )

    partition_by_2 = Window.partitionBy(*partition_by).orderBy("band_2")
    bounds_2 = (
        bounds_2.withColumn(
            "new_lower_bound_incomplete_2",
            F.lag("upper_bound_2", 1, -float('inf')).over(partition_by_2),
        )
        .withColumn(
            "upper_bound_2",
            F.lag("new_lower_bound_incomplete_2", -1, float('inf')).over(partition_by_2),
        )        
        .withColumn(
            "lower_bound_2",
            F.coalesce(F.col("new_lower_bound_incomplete_2"), F.col("lower_bound_2")),
        )
        .drop("new_lower_bound_incomplete_2")
    )

    return bounds_1, bounds_2


def _calculate_numerical_bands_matrix(
    df: DataFrame,
    bounds_1: DataFrame,
    bounds_2: DataFrame,
    num_classes: int,
    filter_pop_ref: Optional[Any] = None,
) -> DataFrame:
    """
    Associate each observation with one band for score_1 and one band for score_2.
    """

    base_keys = [
        "score_name_1",
        "score_name_2",
        "perf_name",
        "seg_name",
        "seg_value",
        "baseline_name",
        "baseline_value",
    ]

    if filter_pop_ref is None:
        keys_1 = [
            F.col("main.score_name_1") == F.col("secondary.score_name_1"),
            F.col("main.score_name_2") == F.col("secondary.score_name_2"),
            F.col("main.perf_name") == F.col("secondary.perf_name"),
            F.col("main.seg_name") == F.col("secondary.seg_name"),
            F.col("main.seg_value") == F.col("secondary.seg_value"),
            F.col("main.baseline_name") == F.col("secondary.baseline_name"),
            F.col("main.baseline_value") == F.col("secondary.baseline_value"),
            F.col("main.score_value_1") <= F.col("secondary.upper_bound_1"),
            F.col("main.score_value_1") > F.col("secondary.lower_bound_1"),
        ]
    else:
        keys_1 = [
            F.col("main.score_name_1") == F.col("secondary.score_name_1"),
            F.col("main.score_name_2") == F.col("secondary.score_name_2"),
            F.col("main.perf_name") == F.col("secondary.perf_name"),
            F.col("main.seg_name") == F.col("secondary.seg_name"),
            F.col("main.seg_value") == F.col("secondary.seg_value"),
            F.col("main.baseline_name") == F.col("secondary.baseline_name"),
            F.col("main.score_value_1") <= F.col("secondary.upper_bound_1"),
            F.col("main.score_value_1") > F.col("secondary.lower_bound_1"),
        ]

    bands_1 = (
        df.alias("main")
        .join(
            bounds_1.alias("secondary"),
            on=keys_1,
            how="left",
        )
        .select(
            F.col("main.*"),
            F.col("secondary.band_1"),
            F.col("secondary.lower_bound_1"),
            F.col("secondary.upper_bound_1"),
        )
    )

    if filter_pop_ref is None:
        keys_2 = [
            F.col("main.score_name_1") == F.col("secondary.score_name_1"),
            F.col("main.score_name_2") == F.col("secondary.score_name_2"),
            F.col("main.perf_name") == F.col("secondary.perf_name"),
            F.col("main.seg_name") == F.col("secondary.seg_name"),
            F.col("main.seg_value") == F.col("secondary.seg_value"),
            F.col("main.baseline_name") == F.col("secondary.baseline_name"),
            F.col("main.baseline_value") == F.col("secondary.baseline_value"),
            F.col("main.score_value_2") <= F.col("secondary.upper_bound_2"),
            F.col("main.score_value_2") > F.col("secondary.lower_bound_2"),
        ]
    else:
        keys_2 = [
            F.col("main.score_name_1") == F.col("secondary.score_name_1"),
            F.col("main.score_name_2") == F.col("secondary.score_name_2"),
            F.col("main.perf_name") == F.col("secondary.perf_name"),
            F.col("main.seg_name") == F.col("secondary.seg_name"),
            F.col("main.seg_value") == F.col("secondary.seg_value"),
            F.col("main.baseline_name") == F.col("secondary.baseline_name"),
            F.col("main.score_value_2") <= F.col("secondary.upper_bound_2"),
            F.col("main.score_value_2") > F.col("secondary.lower_bound_2"),
        ]

    bands_2 = (
        bands_1.alias("main")
        .join(
            bounds_2.alias("secondary"),
            on=keys_2,
            how="left",
        )
        # I CHANGED HERE: select the main columns explicitly to avoid carrying
        # duplicate band_1 / lower_bound_1 / upper_bound_1 names alongside the
        # final aliased versions, which makes later groupBy calls ambiguous.
        .select(
            *[F.col(f"main.{c}").alias(c) for c in base_keys],
            F.col("main.perf_value"),
            F.col("main.score_value_1"),
            F.col("main.score_value_2"),
            F.coalesce(F.col("main.band_1"), F.lit(num_classes + 1)).alias("band_1"),
            F.col("main.lower_bound_1").alias("lower_bound_1"),
            F.col("main.upper_bound_1").alias("upper_bound_1"),
            F.coalesce(F.col("secondary.band_2"), F.lit(num_classes + 1)).alias("band_2"),
            F.col("secondary.lower_bound_2").alias("lower_bound_2"),
            F.col("secondary.upper_bound_2").alias("upper_bound_2"),
        )
    )

    return bands_2


def compute_gains_table_matrix(
    df: DataFrame,
    score_1_list: List[str],
    score_2_list: List[str],
    perf_list: List[str],
    baseline_list: List[str],
    seg_list: Optional[List[str]] = None,
    num_classes: int = 10,
    filter_pop_ref: Optional[Any] = None,
    is_validation: bool = False,
) -> Tuple[DataFrame, DataFrame]:
    """
    Compute a gains table for a matrix formed by score_1 x score_2.
    """
    if seg_list is None:
        seg_list = []

    if not score_1_list:
        raise ValueError("score_1_list cannot be empty.")

    if not score_2_list:
        raise ValueError("score_2_list cannot be empty.")

    if len(seg_list) == 0:
        seg_list = ["overall"]
        df = df.withColumn("overall", F.lit("Sem segmentação").cast("string"))

    stacked = stack_feature_columns_matrix(
        df=df,
        feature_columns_1=score_1_list,
        feature_columns_2=score_2_list,
        baseline_list=baseline_list,
        perf_list=perf_list,
        seg_list=seg_list,
    )

    if is_validation is False:
        stacked = stacked.filter(
            F.col("score_value_1").isNotNull()
            & F.col("score_value_2").isNotNull()
            & F.col("perf_value").isNotNull()
        )

    dev_df = stacked if filter_pop_ref is None else stacked.filter(filter_pop_ref)

    bounds_1, bounds_2 = _calculate_numerical_threshold_bounds_matrix(dev_df, bins=num_classes)
    bands = _calculate_numerical_bands_matrix(
        stacked,
        bounds_1,
        bounds_2,
        num_classes,
        filter_pop_ref,
    )

    partition_columns = [
        "score_name_1",
        "score_name_2",
        "perf_name",
        "seg_name",
        "seg_value",
        "baseline_name",
        "baseline_value",
    ]

    distributions = (
        bands.groupBy(
            *partition_columns,
            "band_1",
            "band_2",
            "lower_bound_1",
            "upper_bound_1",
            "lower_bound_2",
            "upper_bound_2",
        )
        .agg(
            F.count("*").alias("total"),
            F.sum(F.when(F.col("perf_value") == 0, 1).otherwise(0)).alias("good"),
            F.sum(F.when(F.col("perf_value") == 1, 1).otherwise(0)).alias("bad"),
        )
        .withColumn("bad_rate", F.when(F.col("total") > 0, F.round(F.col("bad") / F.col("total"), 4)))
    )

    if filter_pop_ref is None:
        ordering_ref_base = bands
        ordering_columns = partition_columns.copy()
    else:
        # I CHANGED HERE: use the same rationale as compute_gains_table and apply
        # the reference-population filter on the pre-aggregation dataframe, where
        # raw columns like safra_final still exist.
        ordering_ref_base = bands.filter(filter_pop_ref)
        ordering_columns = [
            "score_name_1",
            "score_name_2",
            "perf_name",
            "seg_name",
            "seg_value",
        ]

    ordering_ref = (
        ordering_ref_base.groupBy(*ordering_columns, "band_1", "band_2")
        .agg(
            F.count("*").alias("total_ref"),
            F.sum(F.when(F.col("perf_value") == 1, 1).otherwise(0)).alias("bad_ref"),
        )
        .withColumn("bad_rate_ref", F.when(F.col("total_ref") > 0, F.round(F.col("bad_ref") / F.col("total_ref"), 4)))
    )

    ordering_window = Window.partitionBy(*ordering_columns).orderBy(
        F.col("bad_rate_ref").desc_nulls_last(),
        F.col("band_1").desc_nulls_last(),
        F.col("band_2").desc_nulls_last(),
    )

    ordering_ref = ordering_ref.withColumn("matrix_order", F.row_number().over(ordering_window))

    ordering_join_keys = [
        "score_name_1",
        "score_name_2",
        "perf_name",
        "seg_name",
        "seg_value",
    ]
    if filter_pop_ref is None:
        ordering_join_keys.extend(["baseline_name", "baseline_value"])
    ordering_join_keys.extend(["band_1", "band_2"])

    distributions = (
        distributions.join(ordering_ref, on=ordering_join_keys, how="left")
        .withColumn(
            "matrix_order",
            F.coalesce(
                F.col("matrix_order"),
                (F.col("band_1") - 1) * F.lit(num_classes + 1) + F.col("band_2"),
            ),
        )
    )

    window_spec = Window.partitionBy(*partition_columns)
    cumsum_window = window_spec.orderBy("matrix_order")

    gains_table_complete = (
        distributions.withColumn("total_overall", F.sum("total").over(window_spec))
        .withColumn("total_good", F.sum("good").over(window_spec))
        .withColumn("total_bad", F.sum("bad").over(window_spec))
        .withColumn("dist_good", F.round(F.col("good") / F.col("total_good"), 4))
        .withColumn("dist_bad", F.round(F.col("bad") / F.col("total_bad"), 4))
        .withColumn(
            "woe",
            F.round(
                F.log(
                    F.when(F.col("dist_bad") == 0, None).otherwise(
                        F.col("dist_good") / F.col("dist_bad")
                    )
                ),
                4,
            ),
        )
        .withColumn("iv", F.round((F.col("dist_good") - F.col("dist_bad")) * F.col("woe"), 4))
        .withColumn("iv_total", F.round(F.sum("iv").over(window_spec), 4))
        .withColumn("cum_good", F.sum("good").over(cumsum_window))
        .withColumn("cum_bad", F.sum("bad").over(cumsum_window))
        .withColumn("cum_good_rate", F.round(F.col("cum_good") / F.col("total_good"), 4))
        .withColumn("cum_bad_rate", F.round(F.col("cum_bad") / F.col("total_bad"), 4))
        .withColumn("cum_good_rate_lag1", F.lag("cum_good_rate", 1).over(cumsum_window))
        .withColumn("cum_bad_rate_lag1", F.lag("cum_bad_rate", 1).over(cumsum_window))
        .withColumn("ks", F.round(F.abs(F.col("cum_good_rate") - F.col("cum_bad_rate")), 4))
        .withColumn("max_ks", F.max("ks").over(window_spec))
        .withColumn(
            "aux_auc",
            (
                (F.col("cum_good_rate") - F.coalesce(F.col("cum_good_rate_lag1"), F.lit(0.0)))
                * (F.col("cum_bad_rate") + F.coalesce(F.col("cum_bad_rate_lag1"), F.lit(0.0)))
                * 0.5
            ),
        )
        .withColumn("auc", F.round(F.sum("aux_auc").over(window_spec), 4))
        .withColumn("gini", F.round(2 * F.col("auc") - 1, 4))
        .drop(
            "total_overall",
            "total_good",
            "total_bad",
            "cum_good_rate_lag1",
            "cum_bad_rate_lag1",
            "aux_auc",
        )
    )

    gains_table_summary = (
        gains_table_complete.groupBy(*partition_columns)
        .agg(
            F.max("ks").alias("ks"),
            F.max("gini").alias("gini"),
            F.max("auc").alias("auc"),
            F.max("iv_total").alias("iv_total"),
            F.sum("bad").alias("bad"),
            F.sum("total").alias("total"),
        )
    )

    return gains_table_complete, gains_table_summary


In [5]:

# ------------------------------------------------------------
# Exemplos de uso
# ------------------------------------------------------------
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

df = spark.range(1000).withColumnRenamed("id", "id")

df = df.withColumn('ever60mob6', F.when(F.rand() < 0.5, 1).otherwise(0))
df = df.withColumn('ever90mob12', F.when(F.rand() < 0.4, 1).otherwise(0))
df = df.withColumn('score_serasa', F.rand()*1000)
df = df.withColumn('score_bvs', F.rand()*500)
df = df.withColumn('variable_a', F.rand()*1000)
df = df.withColumn('variable_b', F.rand()*1000)
df = df.withColumn('cliente_alta_renda', F.when(F.rand() < 0.3, 1).otherwise(0))
df = df.withColumn('torcedor_athetico_pr', F.when(F.rand() < 0.4, 1).otherwise(0))
df = df.withColumn('sem_segmento', F.lit('Sem Segmentação'))
df = df.withColumn('safra_final', F.when(F.rand() < 0.7, 'DEV').otherwise('OT1'))

lista_scores = ["score_serasa", "score_bvs"]
lista_baseline = ["safra_final"]
lista_performance = ["ever60mob6", "ever90mob12"]
lista_segmentacao = ["cliente_alta_renda", "torcedor_athetico_pr", "sem_segmento"]

In [6]:
# %pip install openpyxl

from pyspark.sql import functions as F
# from gains_table_matriz import compute_gains_table_matrix

gains_table_matrix_complete, gains_table_matrix_summary = compute_gains_table_matrix(
    df=df,
    score_1_list=["score_serasa"],
    score_2_list=["score_bvs"],
    perf_list=["ever60mob6"],
    baseline_list=["safra_final"],
    # seg_list=["cliente_alta_renda"],
    num_classes=5,
    # filter_pop_ref=(F.col("safra_final") == "DEV"),
    # is_validation=False,
)

# gains_table_matrix_complete.show(1000, truncate=False)
gains_table_matrix_complete.toPandas().to_excel("gains_table_matriz_completa.xlsx", index=False)

# gains_table_matrix_summary.show()
